In [0]:
from pyspark.sql import functions as F
from pyspark.sql import types as T

bronze_wikidata_authors = spark.table(
    "workspace.default.bronze_wikidata_authors_raw"
)

bronze_openlibrary_authors = spark.table(
    "workspace.default.bronze_openlibrary_authors_raw"
)

bronze_openlibrary_works = spark.table(
    "workspace.default.bronze_openlibrary_works_raw"
)

print("Wikidata Bronze:", bronze_wikidata_authors.count())
print("Open Library Authors Bronze:", bronze_openlibrary_authors.count())
print("Open Library Works Bronze:", bronze_openlibrary_works.count())

In [0]:
author_source_mapping = [
    ("Adriana Lisboa", "Q4263049", "OL1527013A", "matched_wikidata_link"),
    ("Ana Maria Gonçalves", "Q16489985", "OL12217460A", "matched_validated_by_known_work"),
    ("Bernardo Carvalho", "Q822971", "OL6495716A", "matched_wikidata_link"),
    ("Carola Saavedra", "Q6538113", "OL5697446A", "matched_validated_manually"),
    ("Conceição Evaristo", "Q16939999", "OL8701565A", "matched_multiple_catalog_records"),
    ("Conceição Evaristo", "Q16939999", "OL5122048A", "matched_multiple_catalog_records"),
    ("Conceição Evaristo", "Q16939999", "OL2940871A", "matched_multiple_catalog_records"),
    ("Geovani Martins", "Q51882320", "OL7633883A", "matched_validated_manually"),
    ("Itamar Vieira Junior", "Q59309236", "OL9081057A", "matched_wikidata_link"),
    ("Jeferson Tenório", "Q112730259", "OL13513413A", "matched_multiple_catalog_records"),
    ("Jeferson Tenório", "Q112730259", "OL11730977A", "matched_multiple_catalog_records"),
    ("Jeferson Tenório", "Q112730259", "OL10432462A", "matched_multiple_catalog_records"),
    ("Julián Fuks", "Q10312919", "OL5533173A", "matched_wikidata_link"),
    ("Milton Hatoum", "Q465693", "OL718270A", "matched_wikidata_link"),
]

silver_author_source_ids = spark.createDataFrame(
    author_source_mapping,
    [
        "author_name_canonical",
        "wikidata_author_id",
        "openlibrary_author_id",
        "match_status",
    ],
)

(
    silver_author_source_ids.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable("workspace.default.silver_author_source_ids")
)

display(
    spark.sql("""
        SELECT
            author_name_canonical,
            COUNT(*) AS openlibrary_ids_count,
            collect_list(openlibrary_author_id) AS openlibrary_author_ids,
            max(match_status) AS match_status
        FROM workspace.default.silver_author_source_ids
        GROUP BY author_name_canonical
        ORDER BY author_name_canonical
    """)
)

In [0]:
wikidata_authors_prepared = (
    bronze_wikidata_authors
    .withColumn(
        "wikidata_author_id",
        F.regexp_extract("author", r"Q\d+$", 0)
    )
    .withColumn(
        "birth_date",
        F.to_date("birthDate")
    )
    .withColumn(
        "citizenship",
        F.trim(F.col("citizenshipLabel"))
    )
    .withColumn(
        "occupation",
        F.trim(F.col("occupationLabel"))
    )
)

wikidata_authors_aggregated = (
    wikidata_authors_prepared
    .groupBy(
        "wikidata_author_id",
        "authorLabel",
        "birth_date",
        "author",
    )
    .agg(
        F.array_sort(
            F.collect_set(
                F.when(F.col("citizenship").isNotNull(), F.col("citizenship"))
            )
        ).alias("citizenships"),
        F.array_sort(
            F.collect_set(
                F.when(F.col("occupation").isNotNull(), F.col("occupation"))
            )
        ).alias("occupations"),
        F.array_sort(
            F.collect_set(
                F.when(
                    F.col("openLibraryAuthorId").isNotNull(),
                    F.col("openLibraryAuthorId")
                )
            )
        ).alias("wikidata_openlibrary_author_ids"),
    )
)

silver_authors = (
    silver_author_source_ids
    .select(
        "author_name_canonical",
        "wikidata_author_id",
        "match_status",
    )
    .dropDuplicates(["wikidata_author_id"])
    .join(
        wikidata_authors_aggregated,
        on="wikidata_author_id",
        how="left",
    )
    .withColumn(
        "has_brazilian_citizenship",
        F.array_contains(F.col("citizenships"), "Brasil"),
    )
    .select(
        "wikidata_author_id",
        "author_name_canonical",
        F.col("authorLabel").alias("wikidata_author_label"),
        "birth_date",
        "citizenships",
        "occupations",
        "has_brazilian_citizenship",
        "wikidata_openlibrary_author_ids",
        F.col("author").alias("wikidata_source_url"),
        "match_status",
    )
)

(
    silver_authors.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable("workspace.default.silver_authors")
)

display(
    spark.sql("""
        SELECT
            author_name_canonical,
            wikidata_author_id,
            birth_date,
            citizenships,
            occupations,
            has_brazilian_citizenship,
            match_status
        FROM workspace.default.silver_authors
        ORDER BY author_name_canonical
    """)
)

In [0]:
from pyspark.sql.window import Window

openlibrary_works_schema = T.StructType([
    T.StructField(
        "entries",
        T.ArrayType(
            T.StructType([
                T.StructField("key", T.StringType(), True),
                T.StructField("title", T.StringType(), True),
                T.StructField("first_publish_date", T.StringType(), True),
                T.StructField("subjects", T.ArrayType(T.StringType()), True),
                T.StructField(
                    "authors",
                    T.ArrayType(
                        T.StructType([
                            T.StructField(
                                "author",
                                T.StructType([
                                    T.StructField("key", T.StringType(), True),
                                ]),
                                True,
                            ),
                            T.StructField(
                                "type",
                                T.StructType([
                                    T.StructField("key", T.StringType(), True),
                                ]),
                                True,
                            ),
                        ])
                    ),
                    True,
                ),
            ])
        ),
        True,
    ),
])

openlibrary_works_parsed = (
    bronze_openlibrary_works
    .withColumn(
        "parsed_json",
        F.from_json(F.col("raw_json"), openlibrary_works_schema),
    )
    .withColumn(
        "work_entry",
        F.explode_outer(F.col("parsed_json.entries")),
    )
    .select(
        "source_author_id",
        "source_url",
        "extracted_at_utc",
        F.col("work_entry.key").alias("work_key"),
        F.col("work_entry.title").alias("work_title"),
        F.col("work_entry.first_publish_date").alias("first_publish_date_raw"),
        F.col("work_entry.subjects").alias("subjects"),
        F.col("work_entry.authors").alias("work_authors_raw"),
    )
    .withColumn(
        "work_id",
        F.regexp_extract("work_key", r"OL\d+W$", 0),
    )
    .withColumn(
        "first_publish_year",
        F.regexp_extract("first_publish_date_raw", r"^\d{4}", 0).cast("int"),
    )
)

silver_work_source_links = (
    openlibrary_works_parsed
    .filter(F.col("work_id") != "")
    .select(
        "work_id",
        "source_author_id",
        "source_url",
        "extracted_at_utc",
    )
    .dropDuplicates()
)

(
    silver_work_source_links.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable("workspace.default.silver_work_source_links")
)

silver_works = (
    openlibrary_works_parsed
    .filter(F.col("work_id") != "")
    .withColumn(
        "row_number",
        F.row_number().over(
            Window.partitionBy("work_id").orderBy(
                F.col("source_author_id").asc()
            )
        ),
    )
    .filter(F.col("row_number") == 1)
    .drop("row_number")
    .select(
        "work_id",
        "work_key",
        "work_title",
        "first_publish_date_raw",
        "first_publish_year",
        "subjects",
        "work_authors_raw",
        "source_url",
        "extracted_at_utc",
    )
)

(
    silver_works.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable("workspace.default.silver_works")
)

print("Registros de origem obra-perfil:", silver_work_source_links.count())
print("Obras únicas:", silver_works.count())

display(
    silver_works.orderBy("work_title")
)

In [0]:
silver_work_author_links = (
    spark.table("workspace.default.silver_work_source_links")
    .join(
        spark.table("workspace.default.silver_author_source_ids"),
        on=(
            F.col("source_author_id")
            == F.col("openlibrary_author_id")
        ),
        how="inner",
    )
    .select(
        "work_id",
        "author_name_canonical",
        "wikidata_author_id",
        "source_author_id",
        "openlibrary_author_id",
        "match_status",
        "source_url",
        "extracted_at_utc",
    )
    .dropDuplicates()
)

(
    silver_work_author_links.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable("workspace.default.silver_work_author_links")
)

print("Relações obra-autor canônico:", silver_work_author_links.count())

display(
    silver_work_author_links
    .groupBy("author_name_canonical")
    .agg(F.countDistinct("work_id").alias("distinct_works"))
    .orderBy("author_name_canonical")
)

In [0]:
silver_work_subjects = (
    spark.table("workspace.default.silver_works")
    .select(
        "work_id",
        F.explode_outer("subjects").alias("subject_raw"),
    )
    .withColumn(
        "subject",
        F.trim(F.col("subject_raw"))
    )
    .filter(
        F.col("subject").isNotNull()
        & (F.col("subject") != "")
    )
    .dropDuplicates(["work_id", "subject"])
)

(
    silver_work_subjects.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable("workspace.default.silver_work_subjects")
)

print("Relações obra-assunto:", silver_work_subjects.count())
print("Assuntos distintos:", silver_work_subjects.select("subject").distinct().count())

display(
    silver_work_subjects
    .groupBy("subject")
    .agg(F.countDistinct("work_id").alias("works_count"))
    .orderBy(F.desc("works_count"), F.asc("subject"))
    .limit(20)
)

In [0]:
import json
from urllib.request import Request, urlopen

def get_json(url):
    request = Request(
        url,
        headers={
            "User-Agent": "mvp-literatura-sul-global-databricks/1.0 (academic data pipeline)"
        },
    )
    with urlopen(request, timeout=30) as response:
        return json.loads(response.read().decode("utf-8"))

test_work_id = (
    spark.table("workspace.default.silver_works")
    .select("work_id")
    .orderBy("work_id")
    .first()["work_id"]
)

test_editions_url = (
    f"https://openlibrary.org/works/{test_work_id}/editions.json?limit=100"
)

test_editions_json = get_json(test_editions_url)

display(
    spark.createDataFrame(
        [
            (
                test_work_id,
                test_editions_url,
                test_editions_json.get("size"),
                len(test_editions_json.get("entries", [])),
            )
        ],
        [
            "work_id",
            "source_url",
            "reported_total_editions",
            "returned_editions",
        ],
    )
)

In [0]:
import time
from datetime import datetime, timezone

work_ids = [
    row["work_id"]
    for row in (
        spark.table("workspace.default.silver_works")
        .select("work_id")
        .orderBy("work_id")
        .collect()
    )
]

editions_records = []
extracted_at_utc = datetime.now(timezone.utc).isoformat()

for work_id in work_ids:
    editions_url = (
        f"https://openlibrary.org/works/{work_id}/editions.json?limit=100"
    )

    try:
        editions_json = get_json(editions_url)

        editions_records.append({
            "work_id": work_id,
            "source_url": editions_url,
            "extracted_at_utc": extracted_at_utc,
            "raw_json": json.dumps(editions_json, ensure_ascii=False),
        })

        print(f"OK: {work_id}")

    except Exception as error:
        print(f"ERRO: {work_id} - {error}")

    time.sleep(1)

print(f"Respostas de edições extraídas: {len(editions_records)}")

In [0]:
from pyspark.sql import Row

df_openlibrary_editions_raw = spark.createDataFrame(
    [Row(**record) for record in editions_records]
)

(
    df_openlibrary_editions_raw.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable("workspace.default.bronze_openlibrary_editions_raw")
)

print(
    "Tabela Bronze de edições criada:",
    "workspace.default.bronze_openlibrary_editions_raw"
)

spark.sql("""
    SELECT COUNT(*) AS total_responses
    FROM workspace.default.bronze_openlibrary_editions_raw
""").show()

In [0]:
from pyspark.sql.functions import (
    col,
    explode,
    from_json,
    lit,
    size,
    when,
)
from pyspark.sql.types import (
    ArrayType,
    MapType,
    StringType,
    StructField,
    StructType,
)

editions_raw_schema = StructType([
    StructField("entries", ArrayType(
        MapType(StringType(), StringType(), valueContainsNull=True),
        containsNull=True
    ), True),
])

df_editions_parsed = (
    spark.table("workspace.default.bronze_openlibrary_editions_raw")
    .withColumn(
        "parsed_json",
        from_json(col("raw_json"), editions_raw_schema),
    )
    .withColumn(
        "edition",
        explode(
            when(
                size(col("parsed_json.entries")) > 0,
                col("parsed_json.entries")
            ).otherwise(lit(None).cast(ArrayType(
                MapType(StringType(), StringType(), valueContainsNull=True)
            )))
        ),
    )
    .withColumn("edition_key", col("edition")["key"])
    .withColumn("title", col("edition")["title"])
    .withColumn("publish_date", col("edition")["publish_date"])
    .withColumn("publish_places", col("edition")["publish_places"])
    .withColumn("publishers", col("edition")["publishers"])
    .withColumn("languages", col("edition")["languages"])
    .select(
        col("work_id"),
        col("source_url"),
        col("extracted_at_utc"),
        col("edition_key"),
        col("title"),
        col("publish_date"),
        col("publish_places"),
        col("publishers"),
        col("languages"),
    )
)

(
    df_editions_parsed.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable("workspace.default.silver_editions")
)

print("Tabela Silver de edições criada:", "workspace.default.silver_editions")

spark.sql("""
    SELECT
        COUNT(*) AS total_rows,
        COUNT(DISTINCT work_id) AS distinct_works,
        COUNT(DISTINCT edition_key) AS distinct_editions
    FROM workspace.default.silver_editions
""").show()

In [0]:
from pyspark.sql.functions import (
    col,
    explode,
    lit,
    size,
    when,
)
from pyspark.sql.types import (
    ArrayType,
    MapType,
    StringType,
    StructField,
    StructType,
)

languages_schema = ArrayType(
    MapType(StringType(), StringType(), valueContainsNull=True),
    containsNull=True
)

df_editions_with_languages = (
    spark.table("workspace.default.silver_editions")
    .withColumn(
        "languages_parsed",
        when(
            col("languages").isNotNull(),
            from_json(col("languages"), languages_schema)
        ).otherwise(lit(None).cast(languages_schema))
    )
    .withColumn(
        "language_entry",
        explode(
            when(
                size(col("languages_parsed")) > 0,
                col("languages_parsed")
            ).otherwise(lit(None).cast(languages_schema))
        )
    )
    .withColumn("language_key", col("language_entry")["key"])
)

df_language_counts = (
    df_editions_with_languages
    .groupBy("language_key")
    .count()
    .orderBy(col("count").desc())
)

display(df_language_counts)

In [0]:
from pyspark.sql.functions import (
    col,
    regexp_extract,
    when,
    expr,
)

df_editions_with_year = (
    spark.table("workspace.default.silver_editions")
    .withColumn(
        "publish_year_raw",
        regexp_extract(col("publish_date"), r"^(\\d{4})", 1)
    )
    .withColumn(
        "publish_year",
        expr("try_cast(publish_year_raw AS INT)")
    )
    .filter(col("publish_year").isNotNull())
)

df_year_counts = (
    df_editions_with_year
    .groupBy("publish_year")
    .count()
    .orderBy(col("publish_year"))
)

display(df_year_counts)

In [0]:
from pyspark.sql.functions import col

sample_dates = (
    spark.table("workspace.default.silver_editions")
    .select("publish_date")
    .filter(col("publish_date").isNotNull())
    .distinct()
    .limit(30)
)

display(sample_dates)

In [0]:
from pyspark.sql.functions import (
    col,
    regexp_extract,
    expr,
)

df_editions_with_year = (
    spark.table("workspace.default.silver_editions")
    .withColumn(
        "publish_year_raw",
        regexp_extract(col("publish_date"), r"(\\d{4})", 1)
    )
    .withColumn(
        "publish_year",
        expr("try_cast(publish_year_raw AS INT)")
    )
    .filter(col("publish_year").isNotNull())
)

df_year_counts = (
    df_editions_with_year
    .groupBy("publish_year")
    .count()
    .orderBy(col("publish_year"))
)

display(df_year_counts)

In [0]:
sample_editions = (
    spark.table("workspace.default.silver_editions")
    .select("work_id", "edition_key", "title", "publish_date", "languages")
    .limit(20)
)

display(sample_editions)

In [0]:
spark.sql("""
    SELECT
        publish_date,
        regexp_extract(publish_date, '(\\\\d{4})', 1) AS extracted_year
    FROM workspace.default.silver_editions
    WHERE publish_date IS NOT NULL
    LIMIT 20
""").show(truncate=False)

In [0]:
from pyspark.sql.functions import (
    col,
    regexp_extract,
    expr,
)

df_editions_with_year = (
    spark.table("workspace.default.silver_editions")
    .withColumn(
        "publish_year_raw",
        regexp_extract(col("publish_date"), r"(\d{4})", 1)
    )
    .withColumn(
        "publish_year",
        expr("try_cast(publish_year_raw AS INT)")
    )
    .filter(col("publish_year").isNotNull())
)

df_year_counts = (
    df_editions_with_year
    .groupBy("publish_year")
    .count()
    .orderBy(col("publish_year"))
)

display(df_year_counts)

In [0]:
(
    df_editions_with_year
    .select(
        "work_id",
        "edition_key",
        "title",
        "publish_date",
        "publish_year",
        "publish_places",
        "publishers",
        "languages",
        "source_url",
        "extracted_at_utc",
    )
    .write
    .format("delta")
    .mode("overwrite")
    .saveAsTable("workspace.default.silver_editions_enriched")
)

print(
    "Tabela Silver enriquecida criada:",
    "workspace.default.silver_editions_enriched"
)

spark.sql("""
    SELECT
        COUNT(*) AS editions_with_valid_year,
        MIN(publish_year) AS earliest_year,
        MAX(publish_year) AS latest_year
    FROM workspace.default.silver_editions_enriched
""").show()

In [0]:
from pyspark.sql.functions import (
    col,
    explode,
    from_json,
)
from pyspark.sql.types import (
    ArrayType,
    MapType,
    StringType,
)

languages_schema = ArrayType(
    MapType(
        StringType(),
        StringType(),
        valueContainsNull=True,
    ),
    containsNull=True,
)

df_edition_languages = (
    spark.table("workspace.default.silver_editions")
    .withColumn(
        "languages_parsed",
        from_json(col("languages"), languages_schema),
    )
    .withColumn(
        "language_entry",
        explode(col("languages_parsed")),
    )
    .select(
        col("edition_key"),
        col("work_id"),
        col("language_entry")["key"].alias("language_key"),
    )
    .filter(col("language_key").isNotNull())
    .dropDuplicates(
        ["edition_key", "language_key"]
    )
)

(
    df_edition_languages.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable("workspace.default.silver_edition_languages")
)

print(
    "Tabela de relações edição-idioma recriada:",
    "workspace.default.silver_edition_languages"
)

spark.sql("""
    SELECT
        COUNT(*) AS edition_language_relations,
        COUNT(DISTINCT edition_key) AS editions_with_language,
        COUNT(DISTINCT language_key) AS distinct_languages,
        COUNT(DISTINCT work_id) AS works_with_language
    FROM workspace.default.silver_edition_languages
""").show()

In [0]:
from pyspark.sql.functions import col

sample_publish_metadata = (
    spark.table("workspace.default.silver_editions")
    .select(
        "edition_key",
        "title",
        "publish_date",
        "publish_places",
        "publishers",
    )
    .filter(
        col("publish_places").isNotNull()
        | col("publishers").isNotNull()
    )
    .limit(20)
)

display(sample_publish_metadata)

In [0]:
from pyspark.sql.functions import (
    col,
    explode,
    from_json,
)
from pyspark.sql.types import (
    ArrayType,
    StringType,
)

places_schema = ArrayType(StringType())

df_edition_places = (
    spark.table("workspace.default.silver_editions")
    .withColumn(
        "publish_places_parsed",
        from_json(col("publish_places"), places_schema),
    )
    .withColumn(
        "publish_place",
        explode(col("publish_places_parsed")),
    )
    .select(
        col("edition_key"),
        col("work_id"),
        col("publish_place"),
    )
    .filter(col("publish_place").isNotNull())
    .dropDuplicates(
        ["edition_key", "publish_place"]
    )
)

(
    df_edition_places.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable("workspace.default.silver_edition_places")
)

print(
    "Tabela de relações edição-local criada:",
    "workspace.default.silver_edition_places"
)

spark.sql("""
    SELECT
        COUNT(*) AS edition_place_relations,
        COUNT(DISTINCT edition_key) AS editions_with_place,
        COUNT(DISTINCT publish_place) AS distinct_places
    FROM workspace.default.silver_edition_places
""").show()

In [0]:
df_edition_publishers = (
    spark.table("workspace.default.silver_editions")
    .withColumn(
        "publishers_parsed",
        from_json(col("publishers"), places_schema),
    )
    .withColumn(
        "publisher",
        explode(col("publishers_parsed")),
    )
    .select(
        col("edition_key"),
        col("work_id"),
        col("publisher"),
    )
    .filter(col("publisher").isNotNull())
    .dropDuplicates(
        ["edition_key", "publisher"]
    )
)

(
    df_edition_publishers.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable("workspace.default.silver_edition_publishers")
)

print(
    "Tabela de relações edição-editora criada:",
    "workspace.default.silver_edition_publishers"
)

spark.sql("""
    SELECT
        COUNT(*) AS edition_publisher_relations,
        COUNT(DISTINCT edition_key) AS editions_with_publisher,
        COUNT(DISTINCT publisher) AS distinct_publishers
    FROM workspace.default.silver_edition_publishers
""").show()

In [0]:
spark.sql("""
    CREATE OR REPLACE TABLE workspace.default.silver_editions_data_quality AS
    SELECT
        COUNT(*) AS total_editions,
        COUNT(DISTINCT work_id) AS total_works,
        COUNT(DISTINCT CASE
            WHEN publish_date IS NOT NULL
             AND regexp_extract(publish_date, '(\\\\d{4})', 1) <> ''
            THEN edition_key
        END) AS editions_with_extractable_year,
        COUNT(DISTINCT CASE
            WHEN languages IS NOT NULL
            THEN edition_key
        END) AS editions_with_language_field,
        COUNT(DISTINCT CASE
            WHEN publish_places IS NOT NULL
            THEN edition_key
        END) AS editions_with_place_field,
        COUNT(DISTINCT CASE
            WHEN publishers IS NOT NULL
            THEN edition_key
        END) AS editions_with_publisher_field
    FROM workspace.default.silver_editions
""")

print(
    "Tabela de qualidade criada:",
    "workspace.default.silver_editions_data_quality"
)

spark.table(
    "workspace.default.silver_editions_data_quality"
).show()

In [0]:
spark.sql("""
    CREATE OR REPLACE TABLE workspace.default.gold_work_edition_profile AS
    WITH edition_count_summary AS (
        SELECT
            work_id,
            COUNT(DISTINCT edition_key) AS total_editions
        FROM workspace.default.silver_editions
        GROUP BY work_id
    ),

    edition_year_summary AS (
        SELECT
            work_id,
            MIN(publish_year) AS earliest_edition_year,
            MAX(publish_year) AS latest_edition_year
        FROM workspace.default.silver_editions_enriched
        GROUP BY work_id
    ),

    language_summary AS (
        SELECT
            work_id,
            COUNT(DISTINCT edition_key) AS editions_with_language,
            COUNT(DISTINCT language_key) AS distinct_languages,
            COUNT(DISTINCT CASE
                WHEN language_key = '/languages/por'
                THEN edition_key
            END) AS portuguese_editions,
            COUNT(DISTINCT CASE
                WHEN language_key = '/languages/eng'
                THEN edition_key
            END) AS english_editions,
            COUNT(DISTINCT CASE
                WHEN language_key = '/languages/spa'
                THEN edition_key
            END) AS spanish_editions,
            COUNT(DISTINCT CASE
                WHEN language_key = '/languages/fre'
                THEN edition_key
            END) AS french_editions,
            COUNT(DISTINCT CASE
                WHEN language_key = '/languages/ger'
                THEN edition_key
            END) AS german_editions
        FROM workspace.default.silver_edition_languages
        GROUP BY work_id
    )

    SELECT
        w.work_id,
        w.work_title,
        w.work_authors_raw,
        COALESCE(ecs.total_editions, 0) AS total_editions,
        eys.earliest_edition_year,
        eys.latest_edition_year,
        COALESCE(ls.editions_with_language, 0) AS editions_with_language,
        COALESCE(ls.distinct_languages, 0) AS distinct_languages,
        COALESCE(ls.portuguese_editions, 0) AS portuguese_editions,
        COALESCE(ls.english_editions, 0) AS english_editions,
        COALESCE(ls.spanish_editions, 0) AS spanish_editions,
        COALESCE(ls.french_editions, 0) AS french_editions,
        COALESCE(ls.german_editions, 0) AS german_editions
    FROM workspace.default.silver_works AS w
    LEFT JOIN edition_count_summary AS ecs
        ON w.work_id = ecs.work_id
    LEFT JOIN edition_year_summary AS eys
        ON w.work_id = eys.work_id
    LEFT JOIN language_summary AS ls
        ON w.work_id = ls.work_id
""")

print(
    "Tabela Gold de perfil obra-edição criada:",
    "workspace.default.gold_work_edition_profile"
)

spark.sql("""
    SELECT
        COUNT(*) AS total_works,
        SUM(total_editions) AS total_editions,
        SUM(portuguese_editions) AS portuguese_editions,
        SUM(english_editions) AS english_editions,
        SUM(spanish_editions) AS spanish_editions,
        SUM(french_editions) AS french_editions,
        SUM(german_editions) AS german_editions
    FROM workspace.default.gold_work_edition_profile
""").show()

In [0]:
spark.sql("""
    CREATE OR REPLACE TABLE workspace.default.gold_work_publication_profile AS
    WITH publisher_summary AS (
        SELECT
            work_id,
            COUNT(DISTINCT edition_key) AS editions_with_publisher,
            COUNT(DISTINCT publisher) AS distinct_publishers
        FROM workspace.default.silver_edition_publishers
        GROUP BY work_id
    ),

    place_summary AS (
        SELECT
            work_id,
            COUNT(DISTINCT edition_key) AS editions_with_place,
            COUNT(DISTINCT publish_place) AS distinct_publish_places
        FROM workspace.default.silver_edition_places
        GROUP BY work_id
    )

    SELECT
        w.work_id,
        w.work_title,
        w.work_authors_raw,
        COALESCE(ps.editions_with_publisher, 0) AS editions_with_publisher,
        COALESCE(ps.distinct_publishers, 0) AS distinct_publishers,
        COALESCE(pls.editions_with_place, 0) AS editions_with_place,
        COALESCE(pls.distinct_publish_places, 0) AS distinct_publish_places
    FROM workspace.default.silver_works AS w
    LEFT JOIN publisher_summary AS ps
        ON w.work_id = ps.work_id
    LEFT JOIN place_summary AS pls
        ON w.work_id = pls.work_id
""")

print(
    "Tabela Gold de perfil editorial criada:",
    "workspace.default.gold_work_publication_profile"
)

spark.sql("""
    SELECT
        COUNT(*) AS total_works,
        SUM(editions_with_publisher) AS editions_with_publisher,
        SUM(editions_with_place) AS editions_with_place,
        SUM(distinct_publishers) AS sum_distinct_publishers_by_work,
        SUM(distinct_publish_places) AS sum_distinct_places_by_work
    FROM workspace.default.gold_work_publication_profile
""").show()

In [0]:
spark.sql("""
    CREATE OR REPLACE TABLE workspace.default.gold_language_distribution AS
    SELECT
        language_key,
        COUNT(DISTINCT edition_key) AS editions,
        COUNT(DISTINCT work_id) AS works
    FROM workspace.default.silver_edition_languages
    GROUP BY language_key
    ORDER BY editions DESC, language_key
""")

spark.sql("""
    CREATE OR REPLACE TABLE workspace.default.gold_publication_year_distribution AS
    SELECT
        publish_year,
        COUNT(DISTINCT edition_key) AS editions,
        COUNT(DISTINCT work_id) AS works
    FROM workspace.default.silver_editions_enriched
    GROUP BY publish_year
    ORDER BY publish_year
""")

spark.sql("""
    CREATE OR REPLACE TABLE workspace.default.gold_publisher_distribution AS
    SELECT
        publisher,
        COUNT(DISTINCT edition_key) AS editions,
        COUNT(DISTINCT work_id) AS works
    FROM workspace.default.silver_edition_publishers
    GROUP BY publisher
    ORDER BY editions DESC, works DESC, publisher
""")

spark.sql("""
    CREATE OR REPLACE TABLE workspace.default.gold_publish_place_distribution AS
    SELECT
        publish_place,
        COUNT(DISTINCT edition_key) AS editions,
        COUNT(DISTINCT work_id) AS works
    FROM workspace.default.silver_edition_places
    GROUP BY publish_place
    ORDER BY editions DESC, works DESC, publish_place
""")

print("Tabelas Gold agregadas criadas com sucesso.")

spark.sql("""
    SELECT 'languages' AS metric, COUNT(*) AS rows
    FROM workspace.default.gold_language_distribution

    UNION ALL

    SELECT 'publication_years' AS metric, COUNT(*) AS rows
    FROM workspace.default.gold_publication_year_distribution

    UNION ALL

    SELECT 'publishers' AS metric, COUNT(*) AS rows
    FROM workspace.default.gold_publisher_distribution

    UNION ALL

    SELECT 'publish_places' AS metric, COUNT(*) AS rows
    FROM workspace.default.gold_publish_place_distribution
""").show()

In [0]:
print("Distribuição de idiomas")
display(
    spark.table("workspace.default.gold_language_distribution")
)

print("Top 15 editoras por número de edições")
display(
    spark.table("workspace.default.gold_publisher_distribution")
    .limit(15)
)

print("Top 15 locais de publicação por número de edições")
display(
    spark.table("workspace.default.gold_publish_place_distribution")
    .limit(15)
)

print("Distribuição anual de edições")
display(
    spark.table("workspace.default.gold_publication_year_distribution")
)

In [0]:
spark.sql("""
    CREATE OR REPLACE TABLE workspace.default.gold_publish_place_distribution_normalized AS
    WITH places_classified AS (
        SELECT
            edition_key,
            work_id,
            publish_place AS publish_place_raw,

            CASE
                WHEN lower(trim(publish_place)) RLIKE '^rio de janeiro(, *rj)?$'
                    THEN 'Rio de Janeiro'

                WHEN lower(trim(publish_place)) RLIKE '^s[aã]o paulo(, *sp)?$'
                    THEN 'São Paulo'

                WHEN lower(trim(publish_place)) RLIKE '^\\[s[aã]o paulo, *brazil\\]$'
                    THEN 'São Paulo'

                WHEN lower(trim(publish_place)) RLIKE '^belo horizonte(, *mg)?$'
                    THEN 'Belo Horizonte'

                WHEN lower(trim(publish_place)) = 'london'
                    THEN 'London'

                WHEN lower(trim(publish_place)) = 'new york'
                    THEN 'New York'

                WHEN lower(trim(publish_place)) = 'edinburgh'
                    THEN 'Edinburgh'

                WHEN lower(trim(publish_place)) = 'brazil'
                    THEN 'Brazil'

                WHEN lower(trim(publish_place)) RLIKE '^[0-9]+ *p$'
                    THEN NULL

                ELSE trim(publish_place)
            END AS publish_place_normalized,

            CASE
                WHEN lower(trim(publish_place)) RLIKE '^[0-9]+ *p$'
                    THEN 'suspect_non_place'

                WHEN trim(publish_place) IS NULL OR trim(publish_place) = ''
                    THEN 'missing_or_empty'

                ELSE 'valid_or_unreviewed'
            END AS place_quality_flag

        FROM workspace.default.silver_edition_places
    )

    SELECT
        publish_place_normalized,
        COUNT(DISTINCT edition_key) AS editions,
        COUNT(DISTINCT work_id) AS works,
        COUNT(DISTINCT publish_place_raw) AS raw_variants
    FROM places_classified
    WHERE publish_place_normalized IS NOT NULL
    GROUP BY publish_place_normalized
    ORDER BY editions DESC, works DESC, publish_place_normalized
""")

print(
    "Tabela Gold de locais normalizados criada:",
    "workspace.default.gold_publish_place_distribution_normalized"
)

spark.sql("""
    SELECT
        place_quality_flag,
        COUNT(*) AS edition_place_relations,
        COUNT(DISTINCT edition_key) AS editions
    FROM (
        SELECT
            edition_key,
            CASE
                WHEN lower(trim(publish_place)) RLIKE '^[0-9]+ *p$'
                    THEN 'suspect_non_place'
                WHEN trim(publish_place) IS NULL OR trim(publish_place) = ''
                    THEN 'missing_or_empty'
                ELSE 'valid_or_unreviewed'
            END AS place_quality_flag
        FROM workspace.default.silver_edition_places
    )
    GROUP BY place_quality_flag
    ORDER BY place_quality_flag
""").show()

display(
    spark.table(
        "workspace.default.gold_publish_place_distribution_normalized"
    ).limit(20)
)

In [0]:
display(
    spark.sql("""
        SELECT
            publish_place_normalized,
            editions,
            works,
            raw_variants
        FROM workspace.default.gold_publish_place_distribution_normalized
        ORDER BY editions DESC, works DESC, publish_place_normalized
        LIMIT 20
    """)
)

In [0]:
spark.sql("""
    CREATE OR REPLACE TABLE workspace.default.gold_publish_place_distribution_normalized_v2 AS
    WITH places_classified AS (
        SELECT
            edition_key,
            work_id,
            publish_place AS publish_place_raw,

            CASE
                WHEN lower(trim(publish_place)) RLIKE '^rio de janeiro(, *rj)?$'
                    THEN 'Rio de Janeiro'

                WHEN lower(trim(publish_place)) RLIKE '^s[aã]o paulo(, *sp)?$'
                    THEN 'São Paulo'

                WHEN lower(trim(publish_place)) RLIKE '^\\[s[aã]o paulo, *brazil\\]$'
                    THEN 'São Paulo'

                WHEN lower(trim(publish_place)) RLIKE '^belo horizonte(, *mg)?(, *brasil)?$'
                    THEN 'Belo Horizonte'

                WHEN lower(trim(publish_place)) = 'london'
                    THEN 'London'

                WHEN lower(trim(publish_place)) = 'new york'
                    THEN 'New York'

                WHEN lower(trim(publish_place)) = 'edinburgh'
                    THEN 'Edinburgh'

                WHEN lower(trim(publish_place)) = 'brazil'
                    THEN 'Brazil'

                WHEN lower(trim(publish_place)) RLIKE '^[0-9]+ *p$'
                    THEN NULL

                ELSE trim(publish_place)
            END AS publish_place_normalized,

            CASE
                WHEN lower(trim(publish_place)) RLIKE '^[0-9]+ *p$'
                    THEN 'suspect_non_place'
                ELSE 'valid_or_unreviewed'
            END AS place_quality_flag

        FROM workspace.default.silver_edition_places
    )

    SELECT
        publish_place_normalized,
        COUNT(DISTINCT edition_key) AS editions,
        COUNT(DISTINCT work_id) AS works,
        COUNT(DISTINCT publish_place_raw) AS raw_variants
    FROM places_classified
    WHERE publish_place_normalized IS NOT NULL
    GROUP BY publish_place_normalized
    ORDER BY editions DESC, works DESC, publish_place_normalized
""")

print(
    "Tabela Gold de locais normalizados v2 criada:",
    "workspace.default.gold_publish_place_distribution_normalized_v2"
)

display(
    spark.table(
        "workspace.default.gold_publish_place_distribution_normalized_v2"
    ).limit(20)
)

In [0]:
spark.sql("""
    CREATE OR REPLACE TABLE workspace.default.gold_publish_place_distribution_normalized_final AS
    WITH cleaned AS (
        SELECT
            edition_key,
            work_id,
            publish_place AS publish_place_raw,

            lower(
                trim(
                    regexp_replace(
                        regexp_replace(
                            publish_place,
                            '[\\\\[\\\\]]',
                            ''
                        ),
                        '\\\\s+',
                        ' '
                    )
                )
            ) AS place_clean
        FROM workspace.default.silver_edition_places
        WHERE publish_place IS NOT NULL
    ),

    classified AS (
        SELECT
            edition_key,
            work_id,
            publish_place_raw,

            CASE
                WHEN place_clean RLIKE '^rio de janeiro(, *rj)?$'
                    THEN 'Rio de Janeiro'

                WHEN place_clean RLIKE '^s[aãa]o paulo(, *sp)?(, *brazil)?$'
                    THEN 'São Paulo'

                WHEN place_clean RLIKE '^belo horizonte(, *mg)?(, *brasil)?$'
                    THEN 'Belo Horizonte'

                WHEN place_clean = 'london'
                    THEN 'London'

                WHEN place_clean = 'new york'
                    THEN 'New York'

                WHEN place_clean = 'edinburgh'
                    THEN 'Edinburgh'

                WHEN place_clean = 'brazil'
                    THEN 'Brazil'

                WHEN place_clean RLIKE '^[0-9]+ *p$'
                    THEN NULL

                ELSE trim(publish_place_raw)
            END AS publish_place_normalized

        FROM cleaned
    )

    SELECT
        publish_place_normalized,
        COUNT(DISTINCT edition_key) AS editions,
        COUNT(DISTINCT work_id) AS works,
        COUNT(DISTINCT publish_place_raw) AS raw_variants
    FROM classified
    WHERE publish_place_normalized IS NOT NULL
    GROUP BY publish_place_normalized
    ORDER BY editions DESC, works DESC, publish_place_normalized
""")

print(
    "Tabela final de locais normalizados criada:",
    "workspace.default.gold_publish_place_distribution_normalized_final"
)

display(
    spark.table(
        "workspace.default.gold_publish_place_distribution_normalized_final"
    ).limit(20)
)

In [0]:
spark.sql("""
    SELECT *
    FROM workspace.default.gold_publish_place_distribution_normalized_final
    WHERE publish_place_normalized LIKE '%[%'
       OR publish_place_normalized LIKE '%]%'
""").show(truncate=False)

In [0]:
spark.sql("""
    CREATE OR REPLACE TABLE workspace.default.gold_work_international_circulation AS
    WITH english_summary AS (
        SELECT
            work_id,
            COUNT(DISTINCT edition_key) AS english_editions
        FROM workspace.default.silver_edition_languages
        WHERE language_key = '/languages/eng'
        GROUP BY work_id
    ),

    place_clean AS (
        SELECT
            edition_key,
            work_id,

            lower(
                trim(
                    regexp_replace(
                        regexp_replace(
                            publish_place,
                            '[\\\\[\\\\]]',
                            ''
                        ),
                        '\\\\s+',
                        ' '
                    )
                )
            ) AS place_clean
        FROM workspace.default.silver_edition_places
        WHERE publish_place IS NOT NULL
    ),

    place_normalized AS (
        SELECT
            edition_key,
            work_id,

            CASE
                WHEN place_clean RLIKE '^rio de janeiro(, *rj)?$'
                    THEN 'Rio de Janeiro'

                WHEN place_clean RLIKE '^s[aãa]o paulo(, *sp)?(, *brazil)?$'
                    THEN 'São Paulo'

                WHEN place_clean RLIKE '^belo horizonte(, *mg)?(, *brasil)?$'
                    THEN 'Belo Horizonte'

                WHEN place_clean = 'london'
                    THEN 'London'

                WHEN place_clean = 'new york'
                    THEN 'New York'

                WHEN place_clean = 'edinburgh'
                    THEN 'Edinburgh'

                WHEN place_clean = 'brazil'
                    THEN 'Brazil'

                WHEN place_clean RLIKE '^[0-9]+ *p$'
                    THEN NULL

                ELSE trim(place_clean)
            END AS publish_place_normalized

        FROM place_clean
    ),

    international_place_summary AS (
        SELECT
            work_id,
            COUNT(DISTINCT edition_key) AS editions_with_international_place,
            COUNT(DISTINCT publish_place_normalized) AS international_places
        FROM place_normalized
        WHERE publish_place_normalized IS NOT NULL
          AND publish_place_normalized NOT IN (
              'Rio de Janeiro',
              'São Paulo',
              'Belo Horizonte',
              'Brazil'
          )
        GROUP BY work_id
    )

    SELECT
        w.work_id,
        w.work_title,
        w.work_authors_raw,

        COALESCE(es.english_editions, 0) AS english_editions,
        COALESCE(ips.editions_with_international_place, 0)
            AS editions_with_international_place,
        COALESCE(ips.international_places, 0)
            AS international_places,

        CASE
            WHEN COALESCE(es.english_editions, 0) > 0
             AND COALESCE(ips.editions_with_international_place, 0) > 0
                THEN 'english_and_international_place'

            WHEN COALESCE(es.english_editions, 0) > 0
                THEN 'english_only'

            WHEN COALESCE(ips.editions_with_international_place, 0) > 0
                THEN 'international_place_only'

            ELSE 'no_cataloged_international_signal'
        END AS international_circulation_signal

    FROM workspace.default.silver_works AS w
    LEFT JOIN english_summary AS es
        ON w.work_id = es.work_id
    LEFT JOIN international_place_summary AS ips
        ON w.work_id = ips.work_id
""")

print(
    "Tabela Gold de circulação internacional criada:",
    "workspace.default.gold_work_international_circulation"
)

spark.sql("""
    SELECT
        international_circulation_signal,
        COUNT(*) AS works,
        SUM(english_editions) AS total_english_editions,
        SUM(editions_with_international_place)
            AS total_editions_with_international_place
    FROM workspace.default.gold_work_international_circulation
    GROUP BY international_circulation_signal
    ORDER BY works DESC, international_circulation_signal
""").show()

In [0]:
display(
    spark.sql("""
        SELECT
            work_id,
            work_title,
            work_authors_raw,
            english_editions,
            editions_with_international_place,
            international_places,
            international_circulation_signal
        FROM workspace.default.gold_work_international_circulation
        WHERE international_circulation_signal =
            'english_and_international_place'
        ORDER BY
            english_editions DESC,
            editions_with_international_place DESC,
            work_title
    """)
)

In [0]:
spark.sql("""
    CREATE OR REPLACE TABLE workspace.default.gold_work_international_circulation_examples AS
    SELECT
        work_id,
        work_title,
        english_editions,
        editions_with_international_place,
        international_places,
        international_circulation_signal
    FROM workspace.default.gold_work_international_circulation
    WHERE international_circulation_signal =
        'english_and_international_place'
    ORDER BY
        english_editions DESC,
        editions_with_international_place DESC,
        work_title,
        work_id
""")

print(
    "Tabela Gold de exemplos internacionais criada:",
    "workspace.default.gold_work_international_circulation_examples"
)

display(
    spark.table(
        "workspace.default.gold_work_international_circulation_examples"
    )
)

In [0]:
spark.sql("""
    SELECT
        COUNT(*) AS total_rows,
        COUNT(DISTINCT work_id) AS distinct_work_ids,
        COUNT(DISTINCT work_title) AS distinct_titles
    FROM workspace.default.gold_work_international_circulation_examples
""").show()

display(
    spark.sql("""
        SELECT
            work_id,
            work_title,
            english_editions,
            editions_with_international_place,
            international_places
        FROM workspace.default.gold_work_international_circulation_examples
        ORDER BY work_title, work_id
    """)
)


In [0]:
spark.sql("""
    CREATE OR REPLACE TABLE workspace.default.gold_mvp_kpis AS
    WITH editions_summary AS (
        SELECT
            COUNT(DISTINCT edition_key) AS total_editions,
            COUNT(DISTINCT work_id) AS works_with_editions
        FROM workspace.default.silver_editions
    ),

    quality_summary AS (
        SELECT
            total_works,
            editions_with_extractable_year,
            editions_with_language_field,
            editions_with_place_field,
            editions_with_publisher_field
        FROM workspace.default.silver_editions_data_quality
    ),

    language_summary AS (
        SELECT
            COUNT(DISTINCT CASE
                WHEN language_key = '/languages/por'
                THEN edition_key
            END) AS portuguese_editions,

            COUNT(DISTINCT CASE
                WHEN language_key = '/languages/eng'
                THEN edition_key
            END) AS english_editions,

            COUNT(DISTINCT CASE
                WHEN language_key = '/languages/spa'
                THEN edition_key
            END) AS spanish_editions,

            COUNT(DISTINCT CASE
                WHEN language_key = '/languages/fre'
                THEN edition_key
            END) AS french_editions,

            COUNT(DISTINCT CASE
                WHEN language_key = '/languages/ger'
                THEN edition_key
            END) AS german_editions,

            COUNT(DISTINCT CASE
                WHEN language_key = '/languages/und'
                THEN edition_key
            END) AS undefined_language_editions
        FROM workspace.default.silver_edition_languages
    ),

    international_summary AS (
        SELECT
            COUNT(DISTINCT CASE
                WHEN english_editions > 0
                THEN work_id
            END) AS works_with_english_edition,

            COUNT(DISTINCT CASE
                WHEN editions_with_international_place > 0
                THEN work_id
            END) AS works_with_international_place,

            COUNT(DISTINCT CASE
                WHEN international_circulation_signal =
                    'english_and_international_place'
                THEN work_id
            END) AS works_with_combined_international_signal
        FROM workspace.default.gold_work_international_circulation
    ),

    place_summary AS (
        SELECT
            MAX(CASE
                WHEN publish_place_normalized = 'Rio de Janeiro'
                THEN editions
            END) AS rio_de_janeiro_editions,

            MAX(CASE
                WHEN publish_place_normalized = 'São Paulo'
                THEN editions
            END) AS sao_paulo_editions,

            MAX(CASE
                WHEN publish_place_normalized = 'Belo Horizonte'
                THEN editions
            END) AS belo_horizonte_editions,

            MAX(CASE
                WHEN publish_place_normalized = 'London'
                THEN editions
            END) AS london_editions,

            MAX(CASE
                WHEN publish_place_normalized = 'New York'
                THEN editions
            END) AS new_york_editions,

            MAX(CASE
                WHEN publish_place_normalized = 'Edinburgh'
                THEN editions
            END) AS edinburgh_editions
        FROM workspace.default.gold_publish_place_distribution_normalized_final
    )

    SELECT
        qs.total_works,
        es.total_editions,
        es.works_with_editions,

        qs.editions_with_extractable_year,
        qs.editions_with_language_field,
        qs.editions_with_place_field,
        qs.editions_with_publisher_field,

        ls.portuguese_editions,
        ls.english_editions,
        ls.spanish_editions,
        ls.french_editions,
        ls.german_editions,
        ls.undefined_language_editions,

        ins.works_with_english_edition,
        ins.works_with_international_place,
        ins.works_with_combined_international_signal,

        ps.rio_de_janeiro_editions,
        ps.sao_paulo_editions,
        ps.belo_horizonte_editions,
        ps.london_editions,
        ps.new_york_editions,
        ps.edinburgh_editions

    FROM editions_summary AS es
    CROSS JOIN quality_summary AS qs
    CROSS JOIN language_summary AS ls
    CROSS JOIN international_summary AS ins
    CROSS JOIN place_summary AS ps
""")

print(
    "Tabela Gold de KPIs criada:",
    "workspace.default.gold_mvp_kpis"
)

display(
    spark.table("workspace.default.gold_mvp_kpis")
)

In [0]:
spark.sql("""
    CREATE OR REPLACE TABLE workspace.default.viz_language_distribution AS
    SELECT
        CASE language_key
            WHEN '/languages/por' THEN 'Português'
            WHEN '/languages/eng' THEN 'Inglês'
            WHEN '/languages/spa' THEN 'Espanhol'
            WHEN '/languages/fre' THEN 'Francês'
            WHEN '/languages/ger' THEN 'Alemão'
            WHEN '/languages/und' THEN 'Não definido'
            ELSE language_key
        END AS language,
        editions,
        works,
        ROUND(
            100.0 * editions / SUM(editions) OVER (),
            1
        ) AS edition_share_pct
    FROM workspace.default.gold_language_distribution
    ORDER BY editions DESC, language
""")

spark.sql("""
    CREATE OR REPLACE TABLE workspace.default.viz_publication_years AS
    SELECT
        publish_year,
        editions,
        works
    FROM workspace.default.gold_publication_year_distribution
    ORDER BY publish_year
""")

spark.sql("""
    CREATE OR REPLACE TABLE workspace.default.viz_top_publishers AS
    SELECT
        publisher,
        editions,
        works,
        ROW_NUMBER() OVER (
            ORDER BY editions DESC, works DESC, publisher
        ) AS ranking
    FROM workspace.default.gold_publisher_distribution
    LIMIT 15
""")

spark.sql("""
    CREATE OR REPLACE TABLE workspace.default.viz_top_places AS
    SELECT
        publish_place_normalized AS publish_place,
        editions,
        works,
        raw_variants,
        ROW_NUMBER() OVER (
            ORDER BY editions DESC, works DESC, publish_place_normalized
        ) AS ranking
    FROM workspace.default.gold_publish_place_distribution_normalized_final
    LIMIT 15
""")

spark.sql("""
    CREATE OR REPLACE TABLE workspace.default.viz_international_circulation AS
    SELECT
        CASE international_circulation_signal
            WHEN 'english_and_international_place'
                THEN 'Edição em inglês e local internacional'
            WHEN 'english_only'
                THEN 'Edição em inglês בלבד'
            WHEN 'international_place_only'
                THEN 'Somente local internacional'
            WHEN 'no_cataloged_international_signal'
                THEN 'Sem sinal internacional catalogado'
            ELSE international_circulation_signal
        END AS circulation_signal,
        COUNT(DISTINCT work_id) AS works,
        ROUND(
            100.0 * COUNT(DISTINCT work_id)
            / SUM(COUNT(DISTINCT work_id)) OVER (),
            1
        ) AS work_share_pct
    FROM workspace.default.gold_work_international_circulation
    GROUP BY international_circulation_signal
    ORDER BY works DESC, circulation_signal
""")

print("Tabelas de visualização criadas com sucesso.")

spark.sql("""
    SELECT 'viz_language_distribution' AS table_name, COUNT(*) AS rows
    FROM workspace.default.viz_language_distribution

    UNION ALL

    SELECT 'viz_publication_years' AS table_name, COUNT(*) AS rows
    FROM workspace.default.viz_publication_years

    UNION ALL

    SELECT 'viz_top_publishers' AS table_name, COUNT(*) AS rows
    FROM workspace.default.viz_top_publishers

    UNION ALL

    SELECT 'viz_top_places' AS table_name, COUNT(*) AS rows
    FROM workspace.default.viz_top_places

    UNION ALL

    SELECT 'viz_international_circulation' AS table_name, COUNT(*) AS rows
    FROM workspace.default.viz_international_circulation
""").show()

In [0]:
validation_sql = """
CREATE OR REPLACE TABLE workspace.default.gold_pipeline_validation AS

SELECT
    'bronze_openlibrary_editions_raw' AS dataset,
    COUNT(*) AS row_count,
    82 AS expected_row_count,
    CASE WHEN COUNT(*) = 82 THEN 'PASS' ELSE 'FAIL' END AS validation_status
FROM workspace.default.bronze_openlibrary_editions_raw

UNION ALL

SELECT
    'silver_works' AS dataset,
    COUNT(*) AS row_count,
    82 AS expected_row_count,
    CASE WHEN COUNT(*) = 82 THEN 'PASS' ELSE 'FAIL' END AS validation_status
FROM workspace.default.silver_works

UNION ALL

SELECT
    'silver_editions' AS dataset,
    COUNT(*) AS row_count,
    116 AS expected_row_count,
    CASE WHEN COUNT(*) = 116 THEN 'PASS' ELSE 'FAIL' END AS validation_status
FROM workspace.default.silver_editions

UNION ALL

SELECT
    'silver_editions_enriched' AS dataset,
    COUNT(*) AS row_count,
    114 AS expected_row_count,
    CASE WHEN COUNT(*) = 114 THEN 'PASS' ELSE 'FAIL' END AS validation_status
FROM workspace.default.silver_editions_enriched

UNION ALL

SELECT
    'silver_edition_languages' AS dataset,
    COUNT(*) AS row_count,
    104 AS expected_row_count,
    CASE WHEN COUNT(*) = 104 THEN 'PASS' ELSE 'FAIL' END AS validation_status
FROM workspace.default.silver_edition_languages

UNION ALL

SELECT
    'silver_edition_places' AS dataset,
    COUNT(*) AS row_count,
    73 AS expected_row_count,
    CASE WHEN COUNT(*) = 73 THEN 'PASS' ELSE 'FAIL' END AS validation_status
FROM workspace.default.silver_edition_places

UNION ALL

SELECT
    'silver_edition_publishers' AS dataset,
    COUNT(*) AS row_count,
    120 AS expected_row_count,
    CASE WHEN COUNT(*) = 120 THEN 'PASS' ELSE 'FAIL' END AS validation_status
FROM workspace.default.silver_edition_publishers

UNION ALL

SELECT
    'gold_work_edition_profile' AS dataset,
    COUNT(*) AS row_count,
    82 AS expected_row_count,
    CASE WHEN COUNT(*) = 82 THEN 'PASS' ELSE 'FAIL' END AS validation_status
FROM workspace.default.gold_work_edition_profile

UNION ALL

SELECT
    'gold_work_publication_profile' AS dataset,
    COUNT(*) AS row_count,
    82 AS expected_row_count,
    CASE WHEN COUNT(*) = 82 THEN 'PASS' ELSE 'FAIL' END AS validation_status
FROM workspace.default.gold_work_publication_profile

UNION ALL

SELECT
    'gold_language_distribution' AS dataset,
    COUNT(*) AS row_count,
    6 AS expected_row_count,
    CASE WHEN COUNT(*) = 6 THEN 'PASS' ELSE 'FAIL' END AS validation_status
FROM workspace.default.gold_language_distribution

UNION ALL

SELECT
    'gold_publication_year_distribution' AS dataset,
    COUNT(*) AS row_count,
    33 AS expected_row_count,
    CASE WHEN COUNT(*) = 33 THEN 'PASS' ELSE 'FAIL' END AS validation_status
FROM workspace.default.gold_publication_year_distribution

UNION ALL

SELECT
    'gold_publisher_distribution' AS dataset,
    COUNT(*) AS row_count,
    67 AS expected_row_count,
    CASE WHEN COUNT(*) = 67 THEN 'PASS' ELSE 'FAIL' END AS validation_status
FROM workspace.default.gold_publisher_distribution

UNION ALL

SELECT
    'gold_publish_place_distribution' AS dataset,
    COUNT(*) AS row_count,
    34 AS expected_row_count,
    CASE WHEN COUNT(*) = 34 THEN 'PASS' ELSE 'FAIL' END AS validation_status
FROM workspace.default.gold_publish_place_distribution

UNION ALL

SELECT
    'gold_work_international_circulation' AS dataset,
    COUNT(*) AS row_count,
    82 AS expected_row_count,
    CASE WHEN COUNT(*) = 82 THEN 'PASS' ELSE 'FAIL' END AS validation_status
FROM workspace.default.gold_work_international_circulation

UNION ALL

SELECT
    'gold_mvp_kpis' AS dataset,
    COUNT(*) AS row_count,
    1 AS expected_row_count,
    CASE WHEN COUNT(*) = 1 THEN 'PASS' ELSE 'FAIL' END AS validation_status
FROM workspace.default.gold_mvp_kpis

UNION ALL

SELECT
    'viz_language_distribution' AS dataset,
    COUNT(*) AS row_count,
    6 AS expected_row_count,
    CASE WHEN COUNT(*) = 6 THEN 'PASS' ELSE 'FAIL' END AS validation_status
FROM workspace.default.viz_language_distribution

UNION ALL

SELECT
    'viz_publication_years' AS dataset,
    COUNT(*) AS row_count,
    33 AS expected_row_count,
    CASE WHEN COUNT(*) = 33 THEN 'PASS' ELSE 'FAIL' END AS validation_status
FROM workspace.default.viz_publication_years

UNION ALL

SELECT
    'viz_top_publishers' AS dataset,
    COUNT(*) AS row_count,
    15 AS expected_row_count,
    CASE WHEN COUNT(*) = 15 THEN 'PASS' ELSE 'FAIL' END AS validation_status
FROM workspace.default.viz_top_publishers

UNION ALL

SELECT
    'viz_top_places' AS dataset,
    COUNT(*) AS row_count,
    15 AS expected_row_count,
    CASE WHEN COUNT(*) = 15 THEN 'PASS' ELSE 'FAIL' END AS validation_status
FROM workspace.default.viz_top_places

UNION ALL

SELECT
    'viz_international_circulation' AS dataset,
    COUNT(*) AS row_count,
    4 AS expected_row_count,
    CASE WHEN COUNT(*) = 4 THEN 'PASS' ELSE 'FAIL' END AS validation_status
FROM workspace.default.viz_international_circulation
"""

spark.sql(validation_sql)

print(
    "Tabela de validação final criada:",
    "workspace.default.gold_pipeline_validation"
)

display(
    spark.table("workspace.default.gold_pipeline_validation")
    .orderBy("dataset")
)

spark.sql("""
    SELECT
        validation_status,
        COUNT(*) AS datasets
    FROM workspace.default.gold_pipeline_validation
    GROUP BY validation_status
    ORDER BY validation_status
""").show()

In [0]:
spark.sql("""
    CREATE OR REPLACE TABLE workspace.default.viz_kpi_cards AS
    SELECT
        1 AS display_order,
        'Obras analisadas' AS kpi_name,
        CAST(total_works AS BIGINT) AS kpi_value,
        'obras' AS unit,
        'Cobertura do corpus analisado' AS description
    FROM workspace.default.gold_mvp_kpis

    UNION ALL

    SELECT
        2 AS display_order,
        'Edições recuperadas' AS kpi_name,
        CAST(total_editions AS BIGINT) AS kpi_value,
        'edições' AS unit,
        'Edições recuperadas da Open Library' AS description
    FROM workspace.default.gold_mvp_kpis

    UNION ALL

    SELECT
        3 AS display_order,
        'Edições em português' AS kpi_name,
        CAST(portuguese_editions AS BIGINT) AS kpi_value,
        'edições' AS unit,
        'Relações edição–idioma classificadas como português' AS description
    FROM workspace.default.gold_mvp_kpis

    UNION ALL

    SELECT
        4 AS display_order,
        'Edições em inglês' AS kpi_name,
        CAST(english_editions AS BIGINT) AS kpi_value,
        'edições' AS unit,
        'Relações edição–idioma classificadas como inglês' AS description
    FROM workspace.default.gold_mvp_kpis

    UNION ALL

    SELECT
        5 AS display_order,
        'Obras com edição em inglês' AS kpi_name,
        CAST(works_with_english_edition AS BIGINT) AS kpi_value,
        'obras' AS unit,
        'Obras com pelo menos uma edição em inglês catalogada' AS description
    FROM workspace.default.gold_mvp_kpis

    UNION ALL

    SELECT
        6 AS display_order,
        'Obras com sinal internacional combinado' AS kpi_name,
        CAST(
            works_with_combined_international_signal
            AS BIGINT
        ) AS kpi_value,
        'obras' AS unit,
        'Obras com edição em inglês e local editorial internacional' AS description
    FROM workspace.default.gold_mvp_kpis

    UNION ALL

    SELECT
        7 AS display_order,
        'Edições com local catalogado' AS kpi_name,
        CAST(editions_with_place_field AS BIGINT) AS kpi_value,
        'edições' AS unit,
        'Cobertura do campo de local de publicação' AS description
    FROM workspace.default.gold_mvp_kpis

    UNION ALL

    SELECT
        8 AS display_order,
        'Edições com ano extraível' AS kpi_name,
        CAST(editions_with_extractable_year AS BIGINT) AS kpi_value,
        'edições' AS unit,
        'Cobertura do campo temporal' AS description
    FROM workspace.default.gold_mvp_kpis
""")

print(
    "Tabela de cartões KPI criada:",
    "workspace.default.viz_kpi_cards"
)

display(
    spark.table("workspace.default.viz_kpi_cards")
    .orderBy("display_order")
)

In [0]:
spark.sql("""
    CREATE OR REPLACE TABLE workspace.default.gold_project_data_inventory AS
    SELECT
        1 AS display_order,
        'Bronze' AS data_layer,
        'bronze_openlibrary_editions_raw' AS table_name,
        'Resposta bruta da API de edições da Open Library' AS purpose,
        'Uma resposta JSON por obra' AS grain,
        82 AS expected_rows

    UNION ALL

    SELECT
        2,
        'Silver',
        'silver_works',
        'Obras normalizadas do corpus',
        'Uma linha por obra',
        82

    UNION ALL

    SELECT
        3,
        'Silver',
        'silver_work_subjects',
        'Relações entre obras e assuntos',
        'Uma linha por relação obra-assunto',
        183

    UNION ALL

    SELECT
        4,
        'Silver',
        'silver_editions',
        'Edições estruturadas da Open Library',
        'Uma linha por edição',
        116

    UNION ALL

    SELECT
        5,
        'Silver',
        'silver_editions_enriched',
        'Edições com ano de publicação extraído',
        'Uma linha por edição com ano válido',
        114

    UNION ALL

    SELECT
        6,
        'Silver',
        'silver_edition_languages',
        'Relações entre edições e idiomas',
        'Uma linha por relação edição-idioma',
        104

    UNION ALL

    SELECT
        7,
        'Silver',
        'silver_edition_places',
        'Relações entre edições e locais de publicação',
        'Uma linha por relação edição-local',
        73

    UNION ALL

    SELECT
        8,
        'Silver',
        'silver_edition_publishers',
        'Relações entre edições e editoras',
        'Uma linha por relação edição-editora',
        120

    UNION ALL

    SELECT
        9,
        'Gold',
        'gold_work_edition_profile',
        'Perfil de edições, idiomas e período por obra',
        'Uma linha por obra',
        82

    UNION ALL

    SELECT
        10,
        'Gold',
        'gold_work_publication_profile',
        'Perfil de editoras e locais por obra',
        'Uma linha por obra',
        82

    UNION ALL

    SELECT
        11,
        'Gold',
        'gold_work_international_circulation',
        'Indicador exploratório de circulação internacional',
        'Uma linha por obra',
        82

    UNION ALL

    SELECT
        12,
        'Gold',
        'gold_mvp_kpis',
        'Indicadores executivos consolidados',
        'Uma linha de métricas do MVP',
        1

    UNION ALL

    SELECT
        13,
        'Validation',
        'gold_pipeline_validation',
        'Auditoria das contagens esperadas',
        'Uma linha por dataset validado',
        20

    UNION ALL

    SELECT
        14,
        'Visualization',
        'viz_kpi_cards',
        'Cartões para dashboard',
        'Uma linha por indicador',
        8

    UNION ALL

    SELECT
        15,
        'Visualization',
        'viz_language_distribution',
        'Distribuição de idiomas para gráfico',
        'Uma linha por idioma',
        6

    UNION ALL

    SELECT
        16,
        'Visualization',
        'viz_publication_years',
        'Distribuição temporal para gráfico',
        'Uma linha por ano',
        33

    UNION ALL

    SELECT
        17,
        'Visualization',
        'viz_top_publishers',
        'Ranking de editoras para gráfico',
        'Uma linha por editora',
        15

    UNION ALL

    SELECT
        18,
        'Visualization',
        'viz_top_places',
        'Ranking de locais normalizados para gráfico',
        'Uma linha por local',
        15

    UNION ALL

    SELECT
        19,
        'Visualization',
        'viz_international_circulation',
        'Distribuição dos sinais internacionais',
        'Uma linha por categoria',
        4
""")

print(
    "Inventário final do projeto criado:",
    "workspace.default.gold_project_data_inventory"
)

display(
    spark.table("workspace.default.gold_project_data_inventory")
    .orderBy("display_order")
)

In [0]:
validation_summary = spark.sql("""
    SELECT
        validation_status,
        COUNT(*) AS datasets
    FROM workspace.default.gold_pipeline_validation
    GROUP BY validation_status
    ORDER BY validation_status
""")

display(validation_summary)